<a href="https://colab.research.google.com/github/twillixa/SL/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import, Load and Cleaning


In [35]:
import os

# --- CONFIG: update these two lines ---
GITHUB_REPO_URL = "https://github.com/twillixa/SL.git"

# --------------------------------------

REPO_NAME = GITHUB_REPO_URL.split("/")[-1].replace(".git", "")

# Clone only if not already cloned (avoids errors on re-run)
if not os.path.exists(f"/content/{REPO_NAME}"):
    os.system(f"git clone {GITHUB_REPO_URL}")
    print(f"Repo cloned: {REPO_NAME}")
else:
    print(f"Repo already cloned, pulling latest changes...")
    os.system(f"cd /content/{REPO_NAME} && git pull")

# Change working directory to repo
os.chdir(f"/content/{REPO_NAME}")
print(f"Working directory: {os.getcwd()}")

# Load dataset
import pandas as pd
df = pd.read_excel("distance_matrix_1.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Repo already cloned, pulling latest changes...
Working directory: /content/SL
Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne","Route de Lavaux 36, 1095 Lutry","Route de la Conversion 261, 1093 La Conversion","Av. des Boveresses 44, 1010 Lausanne","Chemin de la Colice 24, 1023 Crissier","Route de Cossonay 30, 1023 Crissier","Chemin des Lentillières 15, 1023 Crissier","Chemin de Mongevon 28, 1023 Crissier","Chemin de la Gottrause, 11, 1023 Crissier",...,"Route de Genève 25, 1180 Rolle","Rte de Crassier 9, 1262 Eysins","Av. Reverdil 6, 1260 Nyon","Rue de la Morâche 9, 1260 Nyon","Place Bel Air 8, 1260 Nyon","Rue de la Colombière 28, 1260 Nyon","Rue César Soulié 3, 1260 Nyon","Chemin de la Vuarpillière 7, 1260 Nyon","Chemin Des Chalets 9, 1279 Chavannes-de-Bogis","Rue Blavignac 17, 1227 Carouge"
0,"Av. Bergières 10, 1004 Lausanne",0.000,6.276,7.195,7.688,6.372,6.370,5.585,5.777,6.520,...,29.320,41.936,42.259,42.096,42.632,42.708,41.096,42.001,47.538,75.164
1,"Route de Lavaux 36, 1095 Lutry",6.276,0.000,1.683,10.289,13.277,13.274,12.429,12.226,12.661,...,32.935,45.551,45.874,45.711,46.248,46.324,44.711,45.616,51.153,78.246
2,"Route de la Conversion 261, 1093 La Conversion",7.195,1.683,0.000,8.605,17.877,17.874,17.029,16.826,17.261,...,39.509,52.125,52.447,52.284,52.821,52.897,51.285,52.189,57.727,84.373
3,"Av. des Boveresses 44, 1010 Lausanne",7.688,10.289,8.605,0.000,14.458,14.455,13.609,13.406,13.842,...,36.090,48.705,49.028,48.865,49.402,49.478,47.865,48.770,54.307,81.475
4,"Chemin de la Colice 24, 1023 Crissier",6.372,13.277,17.877,14.458,0.000,0.235,1.485,1.678,2.065,...,25.666,38.281,38.604,38.441,38.978,39.054,37.441,38.346,43.883,71.051


## Import of required librairies




In [36]:
!pip install ortools
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "ortools", "-q"])

from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import pandas as pd
import folium
import requests
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from IPython.display import display, HTML


In [37]:
# ─── Fallback Address Library ─────────────────────────────────────────────────
# Run once at the start of the notebook.
# Add any new unresolvable address as a new key/value pair.
# ──────────────────────────────────────────────────────────────────────────────

ADDRESS_FALLBACK = {

    "Ch. de la Place-Verte 34, 1234, Vessy, CH":
        "Chemin de la Place-Verte 32, 1234 Veyrier, Switzerland",

    "Rue de la Maison Carrée 30, 1242, Satigny, CH":
        "Route de la Maison-Carrée 30, 1242 Satigny, Switzerland",

    "Avenue Gratta Paille 2, 1018, Lausanne, CH":
        "Avenue de Gratta-Paille 2, 1018 Lausanne, Switzerland",

    "Rue César-Roux 31, 1001, Lausanne, CH":
        "Rue Dr-César-Roux 31, 1005 Lausanne, Switzerland",


    "Chem. de Riantbosson 19, 1217, Meyrin, CH":
        "Chemin de Riantbosson, 1217 Meyrin, Switzerland",

    "Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH":
        "Chemin Delay 7, 1214 Vernier, Switzerland",

    "EPFL Innovation Park Building C, 1015 Lausanne":
        "EPFL Innovation Park, 1015 Lausanne, Switzerland",

    "Av. Bergières 10, 1004 Lausanne, Switzerland, 1004":
        "Avenue Bergières 10, 1004 Lausanne, Switzerland",

    "Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH":
        "Avenue d'Epenex 4, 1020 Renens, Switzerland",

    "Passage de la Radio, 1211, Genève 1, CH":
        "Passage de la Radio, 1205 Genève, Switzerland",

    "Place de Pont-Rouge, 1212 Lancy, Suisse":
        "Place de Pont-Rouge, 1212 Lancy, Switzerland",

}

## Route 1

In [38]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# ---------- Load distance matrix ----------
file_path = "distance_matrix_1.xlsx"   # change this

df = pd.read_excel("distance_matrix_1.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Choose fixed start and end ----------
start_node = 0   # change this
end_node = len(distance_matrix) - 1   # change this

data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["starts"] = [start_node]
data["ends"] = [end_node]

manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["starts"],
    data["ends"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Open route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Open route:
Av. Bergières 10, 1004 Lausanne -> Route de Lavaux 36, 1095 Lutry -> Route de la Conversion 261, 1093 La Conversion -> Av. des Boveresses 44, 1010 Lausanne -> Route de la Maladière 74, 1022 Chavannes-près-Renens -> EPFL Innovation Park Building C, 1015 Lausanne -> Route de Vallaire 149, 1024 Ecublens VD -> Chemin du Croset 9B, 1024 Ecublens VD -> Chemin de la Gottrause, 11, 1023 Crissier -> Chemin de Mongevon 28, 1023 Crissier -> Chemin des Lentillières 15, 1023 Crissier -> Chemin de la Colice 24, 1023 Crissier -> Route de Cossonay 30, 1023 Crissier -> Rue de l'Industrie 63, 1030 Bussigny -> Rue des Artisans 10, 1026 Echandens -> Rte de Denges 28E, 1027 Lonay -> Rue de Lausanne 45, 1110 Morges -> Rue des Charpentiers 4, 1110 Morges -> Route de la Longeraie 7, 1110 Morges -> Route de Lully 5C, 1131 Tolochenaz -> Chemin Lucien Chevallaz 5, 1170 Aubonne -> Route de Genève 25, 1180 Rolle -> Rue César Soulié 3, 1260 Nyon -> Rue de la Colombière 28, 1260 Nyon -> Place Bel Air 8, 

In [ ]:
# ─── Visualization: Original vs Optimised Route ───────────────────────────────


import folium
import requests
import time

def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Av. Bergières 10, 1004 Lausanne
  OK    Route de Lavaux 36, 1095 Lutry
  OK    Route de la Conversion 261, 1093 La Conversion
  OK    Av. des Boveresses 44, 1010 Lausanne
  OK    Chemin de la Colice 24, 1023 Crissier
  OK    Route de Cossonay 30, 1023 Crissier
  OK    Chemin des Lentillières 15, 1023 Crissier
  OK    Chemin de Mongevon 28, 1023 Crissier
  OK    Chemin de la Gottrause, 11, 1023 Crissier
  OK    Rue de l'Industrie 63, 1030 Bussigny
  OK    Rue des Artisans 10, 1026 Echandens
  OK    Chemin du Croset 9B, 1024 Ecublens VD
  OK    Route de la Maladière 74, 1022 Chavannes-près-Renens
  RETRY  EPFL Innovation Park Building C, 1015 Lausanne
  OK    EPFL Innovation Park Building C, 1015 Lausanne
  OK    Route de Vallaire 149, 1024 Ecublens VD
  OK    Rte de Denges 28E, 1027 Lonay
  OK    Rue de Lausanne 45, 1110 Morges
  OK    Rue des Charpentiers 4, 1110 Morges
  OK    Route de Lully 5C, 1131 Tolochenaz
  OK    Route de la Longeraie 7, 1110 Morge

In [ ]:
def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, not returning to depot
original_order = locations

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Av. Bergières 10, 1004 Lausanne                               ->  Route de Lavaux 36, 1095 Lutry                                   6.276 km
   2. Route de Lavaux 36, 1095 Lutry                                ->  Route de la Conversion 261, 1093 La Conversion                   1.683 km
   3. Route de la Conversion 261, 1093 La Conversion                ->  Av. des Boveresses 44, 1010 Lausanne                             8.605 km
   4. Av. des Boveresses 44, 1010 Lausanne                          ->  Chemin de la Colice 24, 1023 Crissier                           14.458 km
   5. Chemin de la Colice 24, 1023 Crissier                         ->  Route de Cossonay 30, 1023 Crissier                              0.235 km
   6. Route de Cossonay 30, 1023 Crissier                           ->  Chemin des Lentillières 15, 1023 Crissier                        1.454 km
   7. Chemin des Lentillières 15, 1023 Crissier                     ->  Chemin de Mongevon 28, 1023 Crissier 

##Route 2

In [ ]:
df = pd.read_excel("distance_matrix_2.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 32 rows x 33 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne, Switzerland, 1004, ,","Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH","Avenue Gratta Paille 2, 1018, Lausanne, CH","Chemin des Avelines 5, 1004, Lausanne, CH","Rte. des Flumeaux 1, 1008, Prilly, CH","Ch. de la Roche 12, 1020, Renens, CH","Rue neuve 3-5, 1020, Renens VD, CH","Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH","Rue du Marais 17, 1020, Renens VD, CH",...,"Place de la Riponne 10, 1014, Lausanne, CH","Rue César-Roux 31, 1001, Lausanne, CH","Rue Caroline 23, 1003, Lausanne, CH","Rue du Bugnon 21, 1011, Lausanne, CH","Av. du Tribunal-Fédéral 34, 1005, Lausanne, CH","Avenue Guillemin 16, 1009, Pully, CH","Av. de Lavaux 65, 1009, Pully, CH","Av. Charles Ferdinand Ramuz 60, 1009, Pully, CH","Avenue de la gare, 6, 1003, Lausanne, CH","Avenue Georgette 6, 1003, Lausanne, CH"
0,"Av. Bergières 10, 1004 Lausanne, Switzerland, ...",0.00,2.89,1.58,1.03,2.04,3.04,4.48,3.75,3.75,...,1.31,1.93,2.14,2.62,2.63,3.65,4.28,3.91,2.48,2.66
1,"Chemin de Maillefer 59-61, 1052, Le Mont-sur-L...",2.89,0.00,1.67,2.90,4.20,4.91,6.35,5.62,5.62,...,2.80,3.17,3.38,3.87,3.97,5.09,5.72,5.42,4.65,3.93
2,"Avenue Gratta Paille 2, 1018, Lausanne, CH",1.57,1.66,0.00,1.57,2.59,3.59,5.02,4.30,4.30,...,2.70,3.32,3.53,4.01,4.02,5.04,5.67,5.12,3.69,3.87
3,"Chemin des Avelines 5, 1004, Lausanne, CH",1.03,3.15,1.83,0.00,1.27,2.14,3.58,2.85,2.85,...,2.16,2.78,2.99,3.47,3.48,4.50,5.13,4.39,2.96,3.14
4,"Rte. des Flumeaux 1, 1008, Prilly, CH",2.02,3.81,2.49,1.20,0.00,1.13,2.35,1.58,1.58,...,3.15,3.78,3.98,4.47,4.05,5.07,5.70,5.23,3.80,3.98


In [ ]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_2.xlsx"

df = pd.read_excel("distance_matrix_2.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Av. Bergières 10, 1004 Lausanne, Switzerland, 1004 -> Rue de la Borde 41, 1018, Lausanne, CH -> Place de la Riponne 10, 1014, Lausanne, CH -> Place Bel-Air 1, 1003, Lausanne, CH -> Avenue Georgette 6, 1003, Lausanne, CH -> Avenue de la gare, 6, 1003, Lausanne, CH -> Rue Caroline 23, 1003, Lausanne, CH -> Rue du Bugnon 21, 1011, Lausanne, CH -> Rue César-Roux 31, 1001, Lausanne, CH -> Av. du Tribunal-Fédéral 34, 1005, Lausanne, CH -> Avenue Guillemin 16, 1009, Pully, CH -> Av. de Lavaux 65, 1009, Pully, CH -> Av. Charles Ferdinand Ramuz 60, 1009, Pully, CH -> Place de la navigation 14, 1006, Lausanne, CH -> Avenue de Rhodanie 46b, 1007, Lausanne, CH -> Avenue de Rhodanie 40a, 1007, Lausanne, CH -> Chemin de Primerose 11, 1007, Lausanne, CH -> Avenue William-Fraisse 3, 1006, Lausanne, CH -> Avenue d'Ouchy 4, 1006, Lausanne, CH -> Av. Louis-Ruchonnet 16, 1003, Lausanne, CH -> Place de la Gare 11a, 1003, Lausanne, CH -> Avenue de la Gare 44, 1003, Lausanne, CH -> Avenue de Sévelin 3

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map2.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  RETRY  Av. Bergières 10, 1004 Lausanne, Switzerland, 1004
  OK    Av. Bergières 10, 1004 Lausanne, Switzerland, 1004
  OK    Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH
  RETRY  Avenue Gratta Paille 2, 1018, Lausanne, CH
  OK    Avenue Gratta Paille 2, 1018, Lausanne, CH
  OK    Chemin des Avelines 5, 1004, Lausanne, CH
  OK    Rte. des Flumeaux 1, 1008, Prilly, CH
  OK    Chemin de la Roche 3, 1020, Renens VD, CH
  OK    Rue neuve 3-5, 1020, Renens VD, CH
  RETRY  Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH
  OK    Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH
  OK    Rue du Marais 17, 1020, Renens VD, CH
  OK    Rue du Grand-Pré 2b, 1007, Lausanne, CH
  OK    Avenue de Sévelin 32A, 1004, Lausanne, CH
  OK    Avenue de Rhodanie 46b, 1007, Lausanne, CH
  OK    Avenue de Rhodanie 40a, 1007, Lausanne, CH
  OK    Chemin de Primerose 11, 1007, Lausanne, CH
  OK    Place de la navigation 14, 1006, Lausanne, CH
  OK    Avenue William-Fraisse

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────


def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Av. Bergières 10, 1004 Lausanne, Switzerland, 1004            ->  Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH        2.890 km
   2. Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH     ->  Avenue Gratta Paille 2, 1018, Lausanne, CH                       1.670 km
   3. Avenue Gratta Paille 2, 1018, Lausanne, CH                    ->  Chemin des Avelines 5, 1004, Lausanne, CH                        1.570 km
   4. Chemin des Avelines 5, 1004, Lausanne, CH                     ->  Rte. des Flumeaux 1, 1008, Prilly, CH                            1.270 km
   5. Rte. des Flumeaux 1, 1008, Prilly, CH                         ->  Chemin de la Roche 3, 1020, Renens VD, CH                        1.130 km
   6. Chemin de la Roche 3, 1020, Renens VD, CH                     ->  Rue neuve 3-5, 1020, Renens VD, CH                               1.550 km
   7. Rue neuve 3-5, 1020, Renens VD, CH                            ->  Avenue d'Epenex, 4C 1er Etage, 1020, 

##Route 3


In [ ]:
df = pd.read_excel("distance_matrix_3.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Rue Blavignac, 10, 1227, Carouge, CH","Rte de Saint Julien 7, 1227, Carouge GE, CH","Rue Joseph Girard 18, 1227, Carouge GE, CH","Clos de la Fonderie 1, 1227, Carouge GE, CH","Avenue de la Praille 26, 1227, Genève, CH","Avenue de la Praille 50, 1227, Carouge GE, CH","Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH","Rte des Jeunes 9, 1227, Les Acacias, CH",...,"Rue Rodolphe-Toepffer 11 bis, 1206, Genève, CH","Boulevard Helvetique, 8, 1205, Genève, CH","Place de la Taconnerie 3, 1204, Genève, CH","Grand-Rue 25, 1204, Genève, CH","Rue du Port 12, 1204, Genève, CH","Rue Robert-Céard 8, 1204, Genève, CH","Rue du Rhône 62, 1204, Genève, CH","Place du Molard 3, 1204, Genève, CH","Rue du Rhône 42, 1204, Genève, CH","12 Place de la Fusterie, 1204, Genève, CH"
0,"Rue Blavignac 17, 1227, ,",0.00,0.23,1.57,1.32,1.40,0.94,1.46,2.50,1.60,...,2.93,2.80,3.41,3.23,3.69,3.91,3.90,3.98,4.09,3.45
1,"Rue Blavignac, 10, 1227, Carouge, CH",0.19,0.00,1.58,1.33,1.40,0.94,1.46,2.51,1.60,...,2.94,2.80,3.41,3.23,3.70,3.91,3.91,3.98,4.10,3.45
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",1.21,1.03,0.00,0.69,1.71,1.42,1.94,1.69,2.07,...,3.27,2.95,3.61,3.57,3.84,4.06,4.05,4.13,4.24,3.78
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",1.16,0.97,1.12,0.00,1.05,1.29,1.81,2.05,1.95,...,2.96,2.29,2.96,2.07,3.19,3.40,3.40,3.47,3.59,3.67
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",1.68,1.50,2.25,1.60,0.00,0.87,1.38,3.21,1.52,...,2.04,1.73,2.40,2.34,2.63,2.84,2.84,2.91,3.03,2.55


In [ ]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_3.xlsx"

df = pd.read_excel("distance_matrix_3.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, , -> Avenue de la Praille 26, 1227, Genève, CH -> Clos de la Fonderie 1, 1227, Carouge GE, CH -> Rue des Battoirs 7, 1205, Genève, CH -> Av. de la Roseraie 64, 1205, Genève, CH -> Avenue Peschier 41, 1206, Genève, CH -> Route de Florissant 81, 1206, Genève, CH -> Chemin Frank-Thomas 32, 1208, Genève, CH -> Avenue Rosemont 8, 1208, Genève, CH -> Avenue de la Gare des Eaux-Vives 11, 1207, Genève, CH -> Route de Malagnou 28, 1208, Genève, CH -> Rue de Villereuse 22, 1207, Genève, CH -> Rue du Port 12, 1204, Genève, CH -> 12 Place de la Fusterie, 1204, Genève, CH -> Rue du Rhône 42, 1204, Genève, CH -> Place du Molard 3, 1204, Genève, CH -> Rue du Rhône 62, 1204, Genève, CH -> Rue Robert-Céard 8, 1204, Genève, CH -> Grand-Rue 25, 1204, Genève, CH -> Place de la Taconnerie 3, 1204, Genève, CH -> Rue Rodolphe-Toepffer 11 bis, 1206, Genève, CH -> Boulevard Helvetique, 8, 1205, Genève, CH -> Route des Acacias 14, 1227, Les Acacias, CH -> Passage de la Radio, 1211

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map3.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Rue Blavignac 17, 1227, ,
  OK    Rue Blavignac, 10, 1227, Carouge, CH
  OK    Rte de Saint Julien 7, 1227, Carouge GE, CH
  OK    Rue Joseph Girard 18, 1227, Carouge GE, CH
  OK    Clos de la Fonderie 1, 1227, Carouge GE, CH
  OK    Avenue de la Praille 26, 1227, Genève, CH
  OK    Avenue de la praille 50, 1227, Carouge GE, CH
  OK    Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH
  OK    Rte des Jeunes 9, 1227, Les Acacias, CH
  RETRY  Passage de la Radio, 1211, Genève 1, CH
  OK    Passage de la Radio, 1211, Genève 1, CH
  OK    Rue François-Dussaud , 1227, Les Acacias, CH
  OK    Route des Acacias 14, 1227, Les Acacias, CH
  OK    Rue des Battoirs 7, 1205, Genève, CH
  OK    Av. de la Roseraie 64, 1205, Genève, CH
  OK    Avenue Peschier 41, 1206, Genève, CH
  OK    Route de Florissant 81, 1206, Genève, CH
  OK    Route de Malagnou 28, 1208, Genève, CH
  OK    Chemin Frank-Thomas 32, 1208, Genève, CH
  OK    Avenue Rosemont 8, 1208, Genève, CH
  OK

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────


def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Rue Blavignac 17, 1227, ,                                     ->  Rue Blavignac, 10, 1227, Carouge, CH                             0.230 km
   2. Rue Blavignac, 10, 1227, Carouge, CH                          ->  Rte de Saint Julien 7, 1227, Carouge GE, CH                      1.580 km
   3. Rte de Saint Julien 7, 1227, Carouge GE, CH                   ->  Rue Joseph Girard 18, 1227, Carouge GE, CH                       0.690 km
   4. Rue Joseph Girard 18, 1227, Carouge GE, CH                    ->  Clos de la Fonderie 1, 1227, Carouge GE, CH                      1.050 km
   5. Clos de la Fonderie 1, 1227, Carouge GE, CH                   ->  Avenue de la Praille 26, 1227, Genève, CH                        0.870 km
   6. Avenue de la Praille 26, 1227, Genève, CH                     ->  Avenue de la praille 50, 1227, Carouge GE, CH                    0.600 km
   7. Avenue de la praille 50, 1227, Carouge GE, CH                 ->  Esplanade de Pont Rouge 9c, 1212, Gra

##Route 4


In [ ]:
df = pd.read_excel("distance_matrix_4.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 17 rows x 18 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Boulevard James Fazy 8, 1201, Genève, CH","Rue de Chantepoulet, 5, 1201, Genève, CH","Quai du Mont Blanc, 7, 1201, Genève, CH","Quai du Mont Blanc 5, 1201, Genève, CH","Quai du Mont Blanc 3, 1201, Genève, CH","Quai du Mont-Blanc 29, 1201, Genève, CH","Chemin des mines 9, 1203, Genève, CH","Avenue de Sécheron 15, 1202, Geneva, CH","Avenue de Secheron 6, 1202, Genève, CH","Chemin Eugène-Rigot 2, 1202, Genève, CH","Rue de Vermont 62, 1202, Genève, CH","Avenue de la paix 19, 1202, Genève, CH","Avenue Louis-Casai 18, 1209, Genève, CH","Rue de Bourgogne 23, 1203, Genève, CH","Rue de Saint-Jean 30, 1203, Genève, CH","Place de Pont-Rouge, 1212 Lancy, Suisse"
0,"Rue Blavignac 17, 1227, Carouge, CH",0.00,3.53,3.92,4.19,4.18,4.15,4.66,5.42,5.24,5.20,5.57,5.33,6.39,6.15,4.82,4.10,1.98
1,"Boulevard James Fazy 8, 1201, Genève, CH",3.46,0.00,0.56,0.83,0.82,0.79,1.30,2.06,1.87,1.84,2.21,1.71,2.78,2.93,1.59,1.00,3.15
2,"Rue de Chantepoulet, 5, 1201, Genève, CH",3.98,0.54,0.00,0.35,0.34,0.31,0.82,2.00,1.81,1.77,2.14,1.90,2.97,2.84,2.04,1.52,3.66
3,"Quai du Mont Blanc, 7, 1201, Genève, CH",4.26,0.82,0.37,0.00,0.02,0.04,0.53,1.99,1.81,1.77,2.14,9.26,2.97,3.41,2.32,8.26,3.95
4,"Quai du Mont Blanc 5, 1201, Genève, CH",4.21,0.78,0.32,0.23,0.00,0.23,0.71,2.00,1.81,1.77,2.14,9.26,2.97,3.42,2.28,1.76,9.26


In [ ]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_4.xlsx"

df = pd.read_excel("distance_matrix_4.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, Carouge, CH -> Place de Pont-Rouge, 1212 Lancy, Suisse -> Avenue de Sécheron 15, 1202, Geneva, CH -> Chemin Eugène-Rigot 2, 1202, Genève, CH -> Avenue de Secheron 6, 1202, Genève, CH -> Chemin des mines 9, 1203, Genève, CH -> Avenue de la paix 19, 1202, Genève, CH -> Rue de Vermont 62, 1202, Genève, CH -> Avenue Louis-Casai 18, 1209, Genève, CH -> Rue de Bourgogne 23, 1203, Genève, CH -> Rue de Saint-Jean 30, 1203, Genève, CH -> Boulevard James Fazy 8, 1201, Genève, CH -> Quai du Mont Blanc 5, 1201, Genève, CH -> Quai du Mont-Blanc 29, 1201, Genève, CH -> Quai du Mont Blanc, 7, 1201, Genève, CH -> Quai du Mont Blanc 3, 1201, Genève, CH -> Rue de Chantepoulet, 5, 1201, Genève, CH -> Rue Blavignac 17, 1227, Carouge, CH


In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map4.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Rue Blavignac 17, 1227, Carouge, CH
  OK    Boulevard James Fazy 8, 1201, Genève, CH
  OK    Rue de Chantepoulet, 5, 1201, Genève, CH
  OK    Quai du Mont Blanc, 7, 1201, Genève, CH
  OK    Quai du Mont Blanc 5, 1201, Genève, CH
  OK    Quai du Mont Blanc 3, 1201, Genève, CH
  OK    Quai du Mont-Blanc 29, 1201, Genève, CH
  OK    Chemin des mines 9, 1203, Genève, CH
  OK    Avenue de Sécheron 15, 1202, Geneva, CH
  OK    Avenue de Secheron 6, 1202, Genève, CH
  OK    Chemin Eugène-Rigot 2, 1202, Genève, CH
  OK    Rue de Vermont 62, 1202, Genève, CH
  OK    Avenue de la paix 19, 1202, Genève, CH
  OK    Avenue Louis-Casai 18, 1209, Genève, CH
  OK    Rue de Bourgogne 23, 1203, Genève, CH
  OK    Rue de Saint-Jean 30, 1203, Genève, CH
  OK    Place de Pont-Rouge, 1212 Lancy, Suisse

Map saved to route_map4.html


In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────
def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Rue Blavignac 17, 1227, Carouge, CH                           ->  Boulevard James Fazy 8, 1201, Genève, CH                         3.530 km
   2. Boulevard James Fazy 8, 1201, Genève, CH                      ->  Rue de Chantepoulet, 5, 1201, Genève, CH                         0.560 km
   3. Rue de Chantepoulet, 5, 1201, Genève, CH                      ->  Quai du Mont Blanc, 7, 1201, Genève, CH                          0.350 km
   4. Quai du Mont Blanc, 7, 1201, Genève, CH                       ->  Quai du Mont Blanc 5, 1201, Genève, CH                           0.020 km
   5. Quai du Mont Blanc 5, 1201, Genève, CH                        ->  Quai du Mont Blanc 3, 1201, Genève, CH                           0.230 km
   6. Quai du Mont Blanc 3, 1201, Genève, CH                        ->  Quai du Mont-Blanc 29, 1201, Genève, CH                          0.620 km
   7. Quai du Mont-Blanc 29, 1201, Genève, CH                       ->  Chemin des mines 9, 1203, Genève, CH 

##Route 5

In [ ]:
df = pd.read_excel("distance_matrix_5.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 23 rows x 24 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Ch. de la Place-Verte 34, 1234, Vessy, CH","Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH","Chemin Charles-Borgeaud 27, 1213, Onex, CH","Rte de Chancy 85, 1213, Onex, CH","Avenue des Morgines, 8, 1213, Genève, CH","Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH","Avenue de l’Etang 55, 1219, Châtelaine, CH","Chemin de Blandonnet 2, 1214, Vernier, CH",...,"Route de Satigny 50, 1214, Vernier, CH","Route de Satigny 52, 1242, Satigny, CH","Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH","25b route du Bois-de-Bay, 1242, Satigny, CH","Rue de la Maison Carrée 30, 1242, Satigny, CH","Route de l'Aéroport, 21, 1215, Genève, CH","Route de Pré-Bois 29, 1216, Cointrin, CH","L'Ancienne Route 17A, 1218, Le Grand-Saconnex, CH","Route de suisse 160, 1290, Versoix, CH","Ch. Jean-Baptiste Vandelle 3A, 1290, Versoix, CH"
0,"Rue Blavignac 17, 1227, ,",0.00,3.66,3.68,4.34,3.51,3.24,6.40,6.49,7.35,...,10.63,11.10,8.08,11.07,12.99,8.13,7.47,6.81,14.21,12.80
1,"Ch. de la Place-Verte 34, 1234, Vessy, CH",3.95,0.00,6.10,6.64,5.08,5.53,8.07,8.16,9.10,...,12.29,12.76,10.46,12.74,15.29,9.06,9.14,8.28,15.68,14.27
2,"Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates...",3.66,5.98,0.00,2.51,3.27,4.40,8.74,8.83,9.69,...,12.93,12.85,11.14,10.28,10.76,10.58,9.81,9.26,17.39,15.98
3,"Chemin Charles-Borgeaud 27, 1213, Onex, CH",4.29,6.45,2.51,0.00,1.01,1.78,5.00,7.14,8.00,...,11.42,11.34,9.45,8.78,9.25,8.89,8.12,7.57,15.70,14.29
4,"Rte de Chancy 85, 1213, Onex, CH",3.44,5.60,3.45,1.19,0.00,0.92,3.77,6.02,6.88,...,10.16,11.34,8.33,8.78,9.25,7.77,7.01,6.45,14.58,13.17


In [ ]:
# ---------- Load distance matrix ----------
file_path = "distance_matrix_5.xlsx"   # change this

df = pd.read_excel("distance_matrix_5.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, , -> Ch. Jean-Baptiste Vandelle 3A, 1290, Versoix, CH -> Route de suisse 160, 1290, Versoix, CH -> L'Ancienne Route 17A, 1218, Le Grand-Saconnex, CH -> Route de l'Aéroport, 21, 1215, Genève, CH -> Route de Pré-Bois 29, 1216, Cointrin, CH -> 16 chemin des Coquelicots, 1214, Vernier, CH -> Chemin de Blandonnet 2, 1214, Vernier, CH -> Rue des Boudines 6, 1217, Meyrin, CH -> Chem. de Riantbosson 19, 1217, Meyrin, CH -> Rue de Veyrot 39, 1217, Meyrin, CH -> Route de Satigny 50, 1242, Satigny, CH -> Route de Satigny 52, 1242, Satigny, CH -> Rue de la Maison Carrée 30, 1242, Satigny, CH -> 25b route du Bois-de-Bay, 1242, Satigny, CH -> Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH -> Avenue de l’Etang 55, 1219, Châtelaine, CH -> Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH -> Avenue des Morgines, 8, 1213, Genève, CH -> Rte de Chancy 85, 1213, Onex, CH -> Chemin Charles-Borgeaud 27, 1213, Onex, CH -> Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH ->

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map5.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Rue Blavignac 17, 1227, ,
  RETRY  Ch. de la Place-Verte 34, 1234, Vessy, CH
  FAIL  Ch. de la Place-Verte 34, 1234, Vessy, CH
  OK    Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH
  OK    Chemin Charles-Borgeaud 27, 1213, Onex, CH
  OK    Rte de Chancy 85, 1213, Onex, CH
  OK    Avenue des Morgines, 8, 1213, Genève, CH
  OK    Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH
  OK    Avenue de l’Etang 55, 1219, Châtelaine, CH
  OK    Chemin de Blandonnet 2, 1214, Vernier, CH
  OK    16 chemin des Coquelicots, 1214, Vernier, CH
  RETRY  Chem. de Riantbosson 19, 1217, Meyrin, CH
  OK    Chem. de Riantbosson 19, 1217, Meyrin, CH
  OK    Rue des Boudines 6, 1217, Meyrin, CH
  OK    Rue de Veyrot 39, 1217, Meyrin, CH
  OK    Route de Satigny 50, 1242, Satigny, CH
  OK    Route de Satigny 52, 1242, Satigny, CH
  RETRY  Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH
  OK    Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH
  OK    25b route du Bois-

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────

def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Rue Blavignac 17, 1227, ,                                     ->  Ch. de la Place-Verte 34, 1234, Vessy, CH                        3.660 km
   2. Ch. de la Place-Verte 34, 1234, Vessy, CH                     ->  Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH               6.100 km
   3. Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH            ->  Chemin Charles-Borgeaud 27, 1213, Onex, CH                       2.510 km
   4. Chemin Charles-Borgeaud 27, 1213, Onex, CH                    ->  Rte de Chancy 85, 1213, Onex, CH                                 1.010 km
   5. Rte de Chancy 85, 1213, Onex, CH                              ->  Avenue des Morgines, 8, 1213, Genève, CH                         0.920 km
   6. Avenue des Morgines, 8, 1213, Genève, CH                      ->  Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH                  3.300 km
   7. Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH               ->  Avenue de l’Etang 55, 1219, Châtelain

## The following codes are the same routes but calculated by HERE

## Route 1

In [ ]:
df = pd.read_excel("distance_matrix_6.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne","Route de Lavaux 36, 1095 Lutry","Route de la Conversion 261, 1093 La Conversion","Av. des Boveresses 44, 1010 Lausanne","Chemin de la Colice 24, 1023 Crissier","Route de Cossonay 30, 1023 Crissier","Chemin des Lentillières 15, 1023 Crissier","Chemin de Mongevon 28, 1023 Crissier","Chemin de la Gottrause, 11, 1023 Crissier",...,"Route de Genève 25, 1180 Rolle","Rte de Crassier 9, 1262 Eysins","Av. Reverdil 6, 1260 Nyon","Rue de la Morâche 9, 1260 Nyon","Place Bel Air 8, 1260 Nyon","Rue de la Colombière 28, 1260 Nyon","Rue César Soulié 3, 1260 Nyon","Chemin de la Vuarpillière 7, 1260 Nyon","Chemin des Chalets 9, 1279 Chavannes-de-Bogis","Rue Blavignac 17, 1227 Carouge"
0,"Av. Bergières 10, 1004 Lausanne",0.000,6.066,11.401,5.090,6.295,6.371,5.463,5.824,6.362,...,29.633,41.780,42.630,42.295,42.910,40.456,41.303,42.166,47.788,75.077
1,"Route de Lavaux 36, 1095 Lutry",6.343,0.000,1.989,6.746,19.512,19.570,18.650,18.463,18.896,...,33.365,45.512,46.362,46.027,46.642,44.188,45.035,45.898,51.520,78.809
2,"Route de la Conversion 261, 1093 La Conversion",11.926,2.319,0.000,8.191,17.523,17.581,16.661,16.474,16.907,...,39.198,51.345,52.195,51.860,52.475,50.021,50.868,51.731,57.353,84.642
3,"Av. des Boveresses 44, 1010 Lausanne",5.061,6.153,7.856,0.000,14.388,14.446,13.526,13.339,13.772,...,36.063,48.210,49.060,48.725,49.340,46.886,47.733,48.596,54.218,81.507
4,"Chemin de la Colice 24, 1023 Crissier",6.640,18.869,16.550,12.905,0.000,0.154,1.492,1.785,2.218,...,25.743,37.890,38.740,38.405,39.020,36.566,37.413,38.276,43.898,71.187


In [ ]:
# ---------- Load distance matrix ----------
file_path = "distance_matrix_6.xlsx"   # change this

df = pd.read_excel("distance_matrix_6.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Choose fixed start and end ----------
start_node = 0   # change this
end_node = len(distance_matrix) - 1   # change this

data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["starts"] = [start_node]
data["ends"] = [end_node]

manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["starts"],
    data["ends"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Open route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Open route:
Av. Bergières 10, 1004 Lausanne -> Av. des Boveresses 44, 1010 Lausanne -> Route de Lavaux 36, 1095 Lutry -> Route de la Conversion 261, 1093 La Conversion -> EPFL Innovation Park Building C, 1015 Lausanne -> Route de Vallaire 149, 1024 Ecublens VD -> Route de la Maladière 74, 1022 Chavannes-près-Renens -> Chemin du Croset 9B, 1024 Ecublens VD -> Chemin des Lentillières 15, 1023 Crissier -> Chemin de la Gottrause, 11, 1023 Crissier -> Chemin de Mongevon 28, 1023 Crissier -> Chemin de la Colice 24, 1023 Crissier -> Route de Cossonay 30, 1023 Crissier -> Rue de l'Industrie 63, 1030 Bussigny -> Rue des Artisans 10, 1026 Echandens -> Rte de Denges 28E, 1027 Lonay -> Rue de Lausanne 45, 1110 Morges -> Rue des Charpentiers 4, 1110 Morges -> Route de la Longeraie 7, 1110 Morges -> Route de Lully 5C, 1131 Tolochenaz -> Chemin Lucien Chevallaz 5, 1170 Aubonne -> Route de Genève 25, 1180 Rolle -> Rue de la Colombière 28, 1260 Nyon -> Rue César Soulié 3, 1260 Nyon -> Place Bel Air 8, 

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map5.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Av. Bergières 10, 1004 Lausanne
  OK    Route de Lavaux 36, 1095 Lutry
  OK    Route de la Conversion 261, 1093 La Conversion
  OK    Av. des Boveresses 44, 1010 Lausanne
  OK    Chemin de la Colice 24, 1023 Crissier
  OK    Route de Cossonay 30, 1023 Crissier
  OK    Chemin des Lentillières 15, 1023 Crissier
  OK    Chemin de Mongevon 28, 1023 Crissier
  OK    Chemin de la Gottrause, 11, 1023 Crissier
  OK    Rue de l'Industrie 63, 1030 Bussigny
  OK    Rue des Artisans 10, 1026 Echandens
  OK    Chemin du Croset 9B, 1024 Ecublens VD
  OK    Route de la Maladière 74, 1022 Chavannes-près-Renens
  RETRY  EPFL Innovation Park Building C, 1015 Lausanne
  OK    EPFL Innovation Park Building C, 1015 Lausanne
  OK    Route de Vallaire 149, 1024 Ecublens VD
  OK    Rte de Denges 28E, 1027 Lonay
  OK    Rue de Lausanne 45, 1110 Morges
  OK    Rue des Charpentiers 4, 1110 Morges
  OK    Route de Lully 5C, 1131 Tolochenaz
  OK    Route de la Longeraie 7, 1110 Morge

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────

def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Av. Bergières 10, 1004 Lausanne                               ->  Route de Lavaux 36, 1095 Lutry                                   6.066 km
   2. Route de Lavaux 36, 1095 Lutry                                ->  Route de la Conversion 261, 1093 La Conversion                   1.989 km
   3. Route de la Conversion 261, 1093 La Conversion                ->  Av. des Boveresses 44, 1010 Lausanne                             8.191 km
   4. Av. des Boveresses 44, 1010 Lausanne                          ->  Chemin de la Colice 24, 1023 Crissier                           14.388 km
   5. Chemin de la Colice 24, 1023 Crissier                         ->  Route de Cossonay 30, 1023 Crissier                              0.154 km
   6. Route de Cossonay 30, 1023 Crissier                           ->  Chemin des Lentillières 15, 1023 Crissier                        1.568 km
   7. Chemin des Lentillières 15, 1023 Crissier                     ->  Chemin de Mongevon 28, 1023 Crissier 

## Route 2

In [ ]:
df = pd.read_excel("distance_matrix_7.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 32 rows x 33 columns


,DISTANCE MATRIX,"Av. Bergières 10, 1004 Lausanne, Switzerland, 1004","Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH","Avenue Gratta Paille 2, 1018, Lausanne, CH","Chemin des Avelines 5, 1004, Lausanne, CH","Rte. des Flumeaux 1, 1008, Prilly, CH","Chemin de la Roche 3, 1020, Renens VD, CH","Rue neuve 3-5, 1020, Renens VD, CH","Avenue d&#39;Epenex, 4C 1er Etage, 1020, Renens VD, CH","Rue du Marais 17, 1020, Renens VD, CH",...,"Place de la Riponne 10, 1014, Lausanne, CH","Rue César-Roux 31, 1001, Lausanne, CH","Rue Caroline 23, 1003, Lausanne, CH","Rue du Bugnon 21, 1011, Lausanne, CH","Av. du Tribunal-Fédéral 34, 1005, Lausanne, CH","Avenue Guillemin 16, 1009, Pully, CH","Av. de Lavaux 65, 1009, Pully, CH","Av. Charles Ferdinand Ramuz 60, 1009, Pully, CH","Avenue de la gare, 6, 1003, Lausanne, CH","Avenue Georgette 6, 1003, Lausanne, CH"
0,"Av. Bergières 10, 1004 Lausanne, Switzerland, ...",0.000,2.525,1.276,1.134,1.994,3.074,4.557,4.856,3.843,...,1.351,1.952,2.229,2.807,2.838,3.697,4.358,3.854,2.332,2.273
1,"Chemin de Maillefer 59-61, 1052, Le Mont-sur-L...",2.699,0.000,1.678,2.868,3.810,4.890,6.495,6.794,5.697,...,2.989,3.195,3.472,4.050,4.081,5.166,5.827,5.633,4.340,4.052
2,"Avenue Gratta Paille 2, 1018, Lausanne, CH",1.617,1.805,0.000,1.360,2.302,3.382,4.987,5.286,4.189,...,2.356,2.924,3.201,3.779,3.810,4.684,5.345,4.841,3.319,3.260
3,"Chemin des Avelines 5, 1004, Lausanne, CH",1.308,2.822,1.373,0.000,1.117,2.197,3.479,3.778,3.082,...,2.314,3.064,2.867,3.525,3.147,4.042,4.703,4.199,2.677,2.618
4,"Rte. des Flumeaux 1, 1008, Prilly, CH",2.149,4.040,2.591,1.082,0.000,1.248,2.593,2.892,2.156,...,2.811,3.780,3.583,4.241,3.863,4.758,5.419,4.915,3.393,3.334


In [ ]:
# ---------- Load distance matrix ----------
df = pd.read_excel("distance_matrix_7.xlsx", index_col=0)
locations = df.index.tolist()

raw_matrix = df.to_numpy()

scale = 1000
distance_matrix = (raw_matrix * scale).round().astype(int).tolist()

# ---------- Data model ----------
data = {
    "distance_matrix": distance_matrix,
    "num_vehicles": 1,
    "depot": 0,
}

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"],
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data["distance_matrix"][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])

        next_index = solution.Value(routing.NextVar(index))
        next_node = manager.IndexToNode(next_index)

        scaled_cost += distance_matrix[node][next_node]
        exact_cost += raw_matrix[node][next_node]

        index = next_index

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Av. Bergières 10, 1004 Lausanne, Switzerland, 1004 -> Rue de la Borde 41, 1018, Lausanne, CH -> Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH -> Avenue Gratta Paille 2, 1018, Lausanne, CH -> Chemin des Avelines 5, 1004, Lausanne, CH -> Rte. des Flumeaux 1, 1008, Prilly, CH -> Chemin de la Roche 3, 1020, Renens VD, CH -> Rue neuve 3-5, 1020, Renens VD, CH -> Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH -> Rue du Marais 17, 1020, Renens VD, CH -> Rue du Grand-Pré 2b, 1007, Lausanne, CH -> Avenue de Sévelin 32A, 1004, Lausanne, CH -> Chemin de Primerose 11, 1007, Lausanne, CH -> Avenue de Rhodanie 46b, 1007, Lausanne, CH -> Avenue de Rhodanie 40a, 1007, Lausanne, CH -> Place de la navigation 14, 1006, Lausanne, CH -> Avenue William-Fraisse 3, 1006, Lausanne, CH -> Av. Louis-Ruchonnet 16, 1003, Lausanne, CH -> Place de la Gare 11a, 1003, Lausanne, CH -> Avenue de la Gare 44, 1003, Lausanne, CH -> Avenue d'Ouchy 4, 1006, Lausanne, CH -> Avenue de la gare, 6, 1003, L

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map5.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  RETRY  Av. Bergières 10, 1004 Lausanne, Switzerland, 1004
  OK    Av. Bergières 10, 1004 Lausanne, Switzerland, 1004
  OK    Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH
  RETRY  Avenue Gratta Paille 2, 1018, Lausanne, CH
  OK    Avenue Gratta Paille 2, 1018, Lausanne, CH
  OK    Chemin des Avelines 5, 1004, Lausanne, CH
  OK    Rte. des Flumeaux 1, 1008, Prilly, CH
  OK    Chemin de la Roche 3, 1020, Renens VD, CH
  OK    Rue neuve 3-5, 1020, Renens VD, CH
  RETRY  Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH
  OK    Avenue d'Epenex, 4C 1er Etage, 1020, Renens VD, CH
  OK    Rue du Marais 17, 1020, Renens VD, CH
  OK    Rue du Grand-Pré 2b, 1007, Lausanne, CH
  OK    Avenue de Sévelin 32A, 1004, Lausanne, CH
  OK    Avenue de Rhodanie 46b, 1007, Lausanne, CH
  OK    Avenue de Rhodanie 40a, 1007, Lausanne, CH
  OK    Chemin de Primerose 11, 1007, Lausanne, CH
  OK    Place de la navigation 14, 1006, Lausanne, CH
  OK    Avenue William-Fraisse

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────

def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Av. Bergières 10, 1004 Lausanne, Switzerland, 1004            ->  Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH     2525.000 km
   2. Chemin de Maillefer 59-61, 1052, Le Mont-sur-Lausanne, CH     ->  Avenue Gratta Paille 2, 1018, Lausanne, CH                    1678.000 km
   3. Avenue Gratta Paille 2, 1018, Lausanne, CH                    ->  Chemin des Avelines 5, 1004, Lausanne, CH                     1360.000 km
   4. Chemin des Avelines 5, 1004, Lausanne, CH                     ->  Rte. des Flumeaux 1, 1008, Prilly, CH                         1117.000 km
   5. Rte. des Flumeaux 1, 1008, Prilly, CH                         ->  Chemin de la Roche 3, 1020, Renens VD, CH                     1248.000 km
   6. Chemin de la Roche 3, 1020, Renens VD, CH                     ->  Rue neuve 3-5, 1020, Renens VD, CH                            1568.000 km
   7. Rue neuve 3-5, 1020, Renens VD, CH                            ->  Avenue d'Epenex, 4C 1er Etage, 1020, 

## Route 3

In [ ]:
df = pd.read_excel("distance_matrix_8.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 31 rows x 32 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, Carouge, CH","Rue Blavignac, 10, 1227, Carouge, CH","Rte de Saint Julien 7, 1227, Carouge GE, CH","Rue Joseph Girard 18, 1227, Carouge GE, CH","Clos de la Fonderie 1, 1227, Carouge GE, CH","Avenue de la Praille 26, 1227, Genève, CH","Avenue de la praille 50, 1227, Carouge GE, CH","Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH","Rte des Jeunes 9, 1227, Les Acacias, CH",...,"Rue Rodolphe-Toepffer 11 bis, 1206, Genève, CH","Boulevard Helvetique, 8, 1205, Genève, CH","Place de la Taconnerie 3, 1204, Genève, CH","Grand-Rue 25, 1204, Genève, CH","Rue du Port 12, 1204, Genève, CH","Rue Robert-Céard 8, 1204, Genève, CH","Rue du Rhône 62, 1204, Genève, CH","Place du Molard 3, 1204, Genève, CH","Rue du Rhône 42, 1204, Genève, CH","12 Place de la Fusterie, 1204, Genève, CH"
0,"Rue Blavignac 17, 1227, Carouge, CH",0.000,0.213,1.130,1.414,1.364,1.322,1.274,1.545,1.440,...,3.268,2.721,3.186,3.150,3.737,3.835,3.830,3.885,3.995,3.474
1,"Rue Blavignac, 10, 1227, Carouge, CH",0.295,0.000,1.045,1.249,1.199,1.102,1.065,1.336,1.231,...,3.103,2.556,3.021,2.985,3.572,3.670,3.665,3.720,3.830,3.309
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",1.158,1.136,0.000,0.662,1.436,1.802,1.920,2.191,2.086,...,3.473,2.926,3.391,3.401,3.942,4.040,4.035,4.090,4.200,3.725
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",1.085,1.063,1.123,0.000,1.028,1.561,1.671,1.942,1.837,...,2.811,2.264,2.729,3.005,3.280,3.378,3.373,3.428,3.538,3.615
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",1.541,1.309,2.162,0.990,0.000,1.187,1.365,1.636,1.531,...,2.198,1.651,2.116,2.312,2.667,2.765,2.760,2.815,2.925,2.636


In [ ]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_8.xlsx"

df = pd.read_excel("distance_matrix_8.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, Carouge, CH -> Rte de Saint Julien 7, 1227, Carouge GE, CH -> Rue Joseph Girard 18, 1227, Carouge GE, CH -> Clos de la Fonderie 1, 1227, Carouge GE, CH -> Rue des Battoirs 7, 1205, Genève, CH -> Av. de la Roseraie 64, 1205, Genève, CH -> Boulevard Helvetique, 8, 1205, Genève, CH -> Grand-Rue 25, 1204, Genève, CH -> 12 Place de la Fusterie, 1204, Genève, CH -> Rue du Rhône 42, 1204, Genève, CH -> Place du Molard 3, 1204, Genève, CH -> Rue du Rhône 62, 1204, Genève, CH -> Rue Robert-Céard 8, 1204, Genève, CH -> Rue du Port 12, 1204, Genève, CH -> Place de la Taconnerie 3, 1204, Genève, CH -> Rue de Villereuse 22, 1207, Genève, CH -> Avenue de la Gare des Eaux-Vives 11, 1207, Genève, CH -> Chemin Frank-Thomas 32, 1208, Genève, CH -> Avenue Rosemont 8, 1208, Genève, CH -> Route de Florissant 81, 1206, Genève, CH -> Avenue Peschier 41, 1206, Genève, CH -> Route de Malagnou 28, 1208, Genève, CH -> Rue Rodolphe-Toepffer 11 bis, 1206, Genève, CH -> Passage de la 

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map5.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Rue Blavignac 17, 1227, Carouge, CH
  OK    Rue Blavignac, 10, 1227, Carouge, CH
  OK    Rte de Saint Julien 7, 1227, Carouge GE, CH
  OK    Rue Joseph Girard 18, 1227, Carouge GE, CH
  OK    Clos de la Fonderie 1, 1227, Carouge GE, CH
  OK    Avenue de la Praille 26, 1227, Genève, CH
  OK    Avenue de la praille 50, 1227, Carouge GE, CH
  OK    Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH
  OK    Rte des Jeunes 9, 1227, Les Acacias, CH
  RETRY  Passage de la Radio, 1211, Genève 1, CH
  OK    Passage de la Radio, 1211, Genève 1, CH
  OK    Rue François-Dussaud , 1227, Les Acacias, CH
  OK    Route des Acacias 14, 1227, Les Acacias, CH
  OK    Rue des Battoirs 7, 1205, Genève, CH
  OK    Av. de la Roseraie 64, 1205, Genève, CH
  OK    Avenue Peschier 41, 1206, Genève, CH
  OK    Route de Florissant 81, 1206, Genève, CH
  OK    Route de Malagnou 28, 1208, Genève, CH
  OK    Chemin Frank-Thomas 32, 1208, Genève, CH
  OK    Avenue Rosemont 8, 1208, Genèv

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────

def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Rue Blavignac 17, 1227, Carouge, CH                           ->  Rue Blavignac, 10, 1227, Carouge, CH                             0.213 km
   2. Rue Blavignac, 10, 1227, Carouge, CH                          ->  Rte de Saint Julien 7, 1227, Carouge GE, CH                      1.045 km
   3. Rte de Saint Julien 7, 1227, Carouge GE, CH                   ->  Rue Joseph Girard 18, 1227, Carouge GE, CH                       0.662 km
   4. Rue Joseph Girard 18, 1227, Carouge GE, CH                    ->  Clos de la Fonderie 1, 1227, Carouge GE, CH                      1.028 km
   5. Clos de la Fonderie 1, 1227, Carouge GE, CH                   ->  Avenue de la Praille 26, 1227, Genève, CH                        1.187 km
   6. Avenue de la Praille 26, 1227, Genève, CH                     ->  Avenue de la praille 50, 1227, Carouge GE, CH                    0.700 km
   7. Avenue de la praille 50, 1227, Carouge GE, CH                 ->  Esplanade de Pont Rouge 9c, 1212, Gra

## Route 4

In [ ]:
df = pd.read_excel("distance_matrix_9.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 16 rows x 17 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, Carouge, CH","Boulevard James Fazy 8, 1201, Genève, CH","Rue de Chantepoulet, 5, 1201, Genève, CH","Quai du Mont Blanc, 7, 1201, Genève, CH","Quai du Mont Blanc 5, 1201, Genève, CH","Quai du Mont Blanc 3, 1201, Genève, CH","Quai du Mont-Blanc 29, 1201, Genève, CH","Chemin des mines 9, 1203, Genève, CH","Avenue de Sécheron 15, 1202, Geneva, CH","Avenue de Secheron 6, 1202, Genève, CH","Chemin Eugène-Rigot 2, 1202, Genève, CH","Rue de Vermont 62, 1202, Genève, CH","Avenue de la paix 19, 1202, Genève, CH","Avenue Louis-Casai 18, 1209, Genève, CH","Rue de Bourgogne 23, 1203, Genève, CH","Rue de Saint-Jean 30, 1203, Genève, CH"
0,"Rue Blavignac 17, 1227, Carouge, CH",0.000,3.422,3.798,4.160,4.161,4.111,4.667,5.333,5.170,5.223,5.334,5.255,6.235,6.020,4.961,3.595
1,"Boulevard James Fazy 8, 1201, Genève, CH",3.449,0.000,0.471,0.833,0.834,0.784,1.340,2.006,1.843,1.896,2.007,1.928,2.908,2.686,1.627,0.960
2,"Rue de Chantepoulet, 5, 1201, Genève, CH",3.919,0.523,0.000,0.362,0.363,0.313,0.869,1.939,1.776,1.829,1.940,1.886,2.866,2.830,1.986,1.430
3,"Quai du Mont Blanc, 7, 1201, Genève, CH",4.262,0.866,0.343,0.000,0.001,0.049,0.574,1.925,1.777,1.830,1.941,1.903,2.883,3.173,2.329,1.773
4,"Quai du Mont Blanc 5, 1201, Genève, CH",4.261,0.865,0.342,0.001,0.000,0.050,0.573,1.924,1.776,1.829,1.940,1.904,2.884,3.172,2.328,1.772


In [ ]:
# ---------- Load distance matrix from Excel ----------
file_path = "distance_matrix_9.xlsx"

df = pd.read_excel("distance_matrix_9.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, Carouge, CH -> Rue de Saint-Jean 30, 1203, Genève, CH -> Rue de Bourgogne 23, 1203, Genève, CH -> Avenue Louis-Casai 18, 1209, Genève, CH -> Avenue de la paix 19, 1202, Genève, CH -> Rue de Vermont 62, 1202, Genève, CH -> Chemin Eugène-Rigot 2, 1202, Genève, CH -> Avenue de Secheron 6, 1202, Genève, CH -> Avenue de Sécheron 15, 1202, Geneva, CH -> Chemin des mines 9, 1203, Genève, CH -> Quai du Mont-Blanc 29, 1201, Genève, CH -> Quai du Mont Blanc 3, 1201, Genève, CH -> Quai du Mont Blanc 5, 1201, Genève, CH -> Quai du Mont Blanc, 7, 1201, Genève, CH -> Rue de Chantepoulet, 5, 1201, Genève, CH -> Boulevard James Fazy 8, 1201, Genève, CH -> Rue Blavignac 17, 1227, Carouge, CH


In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map5.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Rue Blavignac 17, 1227, Carouge, CH
  OK    Boulevard James Fazy 8, 1201, Genève, CH
  OK    Rue de Chantepoulet, 5, 1201, Genève, CH
  OK    Quai du Mont Blanc, 7, 1201, Genève, CH
  OK    Quai du Mont Blanc 5, 1201, Genève, CH
  OK    Quai du Mont Blanc 3, 1201, Genève, CH
  OK    Quai du Mont-Blanc 29, 1201, Genève, CH
  OK    Chemin des mines 9, 1203, Genève, CH
  OK    Avenue de Sécheron 15, 1202, Geneva, CH
  OK    Avenue de Secheron 6, 1202, Genève, CH
  OK    Chemin Eugène-Rigot 2, 1202, Genève, CH
  OK    Rue de Vermont 62, 1202, Genève, CH
  OK    Avenue de la paix 19, 1202, Genève, CH
  OK    Avenue Louis-Casai 18, 1209, Genève, CH
  OK    Rue de Bourgogne 23, 1203, Genève, CH
  OK    Rue de Saint-Jean 30, 1203, Genève, CH

Map saved to route_map5.html


In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────

def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Rue Blavignac 17, 1227, Carouge, CH                           ->  Boulevard James Fazy 8, 1201, Genève, CH                         3.422 km
   2. Boulevard James Fazy 8, 1201, Genève, CH                      ->  Rue de Chantepoulet, 5, 1201, Genève, CH                         0.471 km
   3. Rue de Chantepoulet, 5, 1201, Genève, CH                      ->  Quai du Mont Blanc, 7, 1201, Genève, CH                          0.362 km
   4. Quai du Mont Blanc, 7, 1201, Genève, CH                       ->  Quai du Mont Blanc 5, 1201, Genève, CH                           0.001 km
   5. Quai du Mont Blanc 5, 1201, Genève, CH                        ->  Quai du Mont Blanc 3, 1201, Genève, CH                           0.050 km
   6. Quai du Mont Blanc 3, 1201, Genève, CH                        ->  Quai du Mont-Blanc 29, 1201, Genève, CH                          0.623 km
   7. Quai du Mont-Blanc 29, 1201, Genève, CH                       ->  Chemin des mines 9, 1203, Genève, CH 

## Route 5

In [ ]:
df = pd.read_excel("Distance_matrix_10.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 23 rows x 24 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, Carouge, CH","Ch. de la Place-Verte 34, 1234, Vessy, CH","Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH","Chemin Charles-Borgeaud 27, 1213, Onex, CH","Rte de Chancy 85, 1213, Onex, CH","Avenue des Morgines, 8, 1213, Genève, CH","Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH","Avenue de l’Etang 55, 1219, Châtelaine, CH","Chemin de Blandonnet 2, 1214, Vernier, CH",...,"Route de Satigny 50, 1242, Satigny, CH","Route de Satigny 52, 1242, Satigny, CH","Losinger Quarz&#39;Up Chemin Delay, 7, 1214, Vernier, CH","25b route du Bois-de-Bay, 1242, Satigny, CH","Rue de la Maison Carrée 30, 1242, Satigny, CH","Route de l&#39;Aéroport, 21, 1215, Genève, CH","Route de Pré-Bois 29, 1216, Cointrin, CH","L'Ancienne Route 17A, 1218, Le Grand-Saconnex, CH","Route de suisse 160, 1290, Versoix, CH","Ch. Jean-Baptiste Vandelle 3A, 1290, Versoix, CH"
0,"Rue Blavignac 17, 1227, Carouge, CH",0.000,3.064,4.014,4.481,5.446,4.016,6.730,6.301,7.801,...,11.165,11.201,9.037,11.454,11.896,14.645,8.794,8571,14529,13619
1,"Ch. de la Place-Verte 34, 1234, Vessy, CH",3.158,0.000,5.490,6.634,7.339,5.909,8.623,8.194,9.694,...,13.058,13.094,10.930,13.347,13.789,11.460,10.687,11106,16674,15764
2,"Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates...",3.801,5.919,0.000,4.579,7.124,4.775,9.936,9.351,8.844,...,12.208,12.244,10.080,12.497,11.119,10.706,11.143,14056,19242,18332
3,"Chemin Charles-Borgeaud 27, 1213, Onex, CH",4.473,7.133,4.568,0.000,1.222,2.026,4.954,4.525,7.898,...,11.262,11.298,9.134,8.790,7.708,9.760,7.018,13110,18296,17386
4,"Rte de Chancy 85, 1213, Onex, CH",4.053,6.713,4.424,2.987,0.000,1.003,3.732,3.303,4.803,...,8.167,8.203,6.039,8.456,8.898,6.569,5.796,7513,15221,14311


In [ ]:
# ---------- Load distance matrix from Excel ----------
file_path = "Distance_matrix_10.xlsx"

df = pd.read_excel("Distance_matrix_10.xlsx", index_col=0)
distance_matrix = df.to_numpy().tolist()
locations = df.index.tolist()

# ---------- Data model ----------
data = {}
data["distance_matrix"] = distance_matrix
data["num_vehicles"] = 1
data["depot"] = 0   # starting/ending node index

# ---------- Routing model ----------
manager = pywrapcp.RoutingIndexManager(
    len(data["distance_matrix"]),
    data["num_vehicles"],
    data["depot"]
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(data["distance_matrix"][from_node][to_node])

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# ---------- Search parameters ----------
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)
search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
)
search_parameters.time_limit.seconds = 10

# ---------- Solve ----------
solution = routing.SolveWithParameters(search_parameters)

# ---------- Print solution ----------
if solution:
    index = routing.Start(0)
    route = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(locations[node])
        previous_index = index
        index = solution.Value(routing.NextVar(index))

    route.append(locations[manager.IndexToNode(index)])

    print("Route:")
    print(" -> ".join(route))
else:
    print("No solution found.")

Route:
Rue Blavignac 17, 1227, Carouge, CH -> Ch. de la Place-Verte 34, 1234, Vessy, CH -> Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH -> Chemin Charles-Borgeaud 27, 1213, Onex, CH -> Rte de Chancy 85, 1213, Onex, CH -> Avenue des Morgines, 8, 1213, Genève, CH -> Avenue de l’Etang 55, 1219, Châtelaine, CH -> Chemin de Blandonnet 2, 1214, Vernier, CH -> 16 chemin des Coquelicots, 1214, Vernier, CH -> Route de Pré-Bois 29, 1216, Cointrin, CH -> L'Ancienne Route 17A, 1218, Le Grand-Saconnex, CH -> Ch. Jean-Baptiste Vandelle 3A, 1290, Versoix, CH -> Route de suisse 160, 1290, Versoix, CH -> Route de l'Aéroport, 21, 1215, Genève, CH -> Chem. de Riantbosson 19, 1217, Meyrin, CH -> Rue des Boudines 6, 1217, Meyrin, CH -> Rue de Veyrot 39, 1217, Meyrin, CH -> Route de Satigny 52, 1242, Satigny, CH -> Route de Satigny 50, 1242, Satigny, CH -> Rue de la Maison Carrée 30, 1242, Satigny, CH -> 25b route du Bois-de-Bay, 1242, Satigny, CH -> Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, C

In [ ]:
def geocode(address: str) -> tuple[float, float] | None:
    """Return (lat, lon) for an address string, or None on failure."""
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    headers = {"User-Agent": "delivery-route-visualizer/1.0"}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=5)
        data = r.json()
        if data:
            return float(data[0]["lat"]), float(data[0]["lon"])
    except Exception:
        pass
    return None

print("Geocoding locations...")
coord_cache: dict[str, tuple[float, float]] = {}

for addr in locations:
    if addr in coord_cache:
        continue
    result = geocode(addr)
    time.sleep(1.1)
    if result is None and addr in ADDRESS_FALLBACK:
        print(f"  RETRY  {addr}")
        result = geocode(ADDRESS_FALLBACK[addr])
        time.sleep(1.1)
    coord_cache[addr] = result
    print(f"  {'OK  ' if result else 'FAIL'}  {addr}")

valid_locations = [a for a in locations if coord_cache.get(a)]
valid_route     = [a for a in route     if coord_cache.get(a)]

all_coords = [coord_cache[a] for a in valid_locations]
centre_lat = sum(c[0] for c in all_coords) / len(all_coords)
centre_lon = sum(c[1] for c in all_coords) / len(all_coords)

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=12, tiles="CartoDB dark_matter")

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_locations],
    color="#4A90E2", weight=2.5, opacity=0.7,
    dash_array="8 6", tooltip="Original order (matrix)"
).add_to(m)

folium.PolyLine(
    locations=[coord_cache[a] for a in valid_route],
    color="#F5A623", weight=3, opacity=0.9,
    tooltip="Optimised route"
).add_to(m)

for addr in valid_locations:
    lat, lon = coord_cache[addr]
    is_depot = (addr == valid_locations[0])
    folium.CircleMarker(
        location=[lat, lon],
        radius=7 if is_depot else 5,
        color="#FFFFFF", fill=True,
        fill_color="#F5A623" if is_depot else "#FFFFFF",
        fill_opacity=1.0, weight=2,
        tooltip=addr
    ).add_to(m)

legend_html = """
<div style="
    position: fixed; bottom: 30px; left: 30px; z-index: 1000;
    background: rgba(20,20,20,0.85); padding: 12px 16px;
    border-radius: 8px; border: 1px solid #444;
    font-family: monospace; font-size: 12px; color: #fff; line-height: 1.8;">
  <b>Route comparison</b><br>
  <span style="color:#4A90E2;">&#9473;&#9473; &#9473;&#9473;</span>&nbsp; Original order (matrix)<br>
  <span style="color:#F5A623;">&#9473;&#9473;&#9473;&#9473;&#9473;</span>&nbsp; Optimised route<br>
  <span style="color:#fff;">&#11044;</span>&nbsp; Stop &nbsp;&nbsp;
  <span style="color:#F5A623;">&#11044;</span>&nbsp; Depot
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

output_file = "route_map5.html"
m.save(output_file)
print(f"\nMap saved to {output_file}")

try:
    from IPython.display import display
    display(m)
except ImportError:
    print("Open route_map.html in your browser to view the map.")

Geocoding locations...
  OK    Rue Blavignac 17, 1227, Carouge, CH
  RETRY  Ch. de la Place-Verte 34, 1234, Vessy, CH
  FAIL  Ch. de la Place-Verte 34, 1234, Vessy, CH
  OK    Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH
  OK    Chemin Charles-Borgeaud 27, 1213, Onex, CH
  OK    Rte de Chancy 85, 1213, Onex, CH
  OK    Avenue des Morgines, 8, 1213, Genève, CH
  OK    Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH
  OK    Avenue de l’Etang 55, 1219, Châtelaine, CH
  OK    Chemin de Blandonnet 2, 1214, Vernier, CH
  OK    16 chemin des Coquelicots, 1214, Vernier, CH
  RETRY  Chem. de Riantbosson 19, 1217, Meyrin, CH
  OK    Chem. de Riantbosson 19, 1217, Meyrin, CH
  OK    Rue des Boudines 6, 1217, Meyrin, CH
  OK    Rue de Veyrot 39, 1217, Meyrin, CH
  OK    Route de Satigny 50, 1242, Satigny, CH
  OK    Route de Satigny 52, 1242, Satigny, CH
  RETRY  Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH
  OK    Losinger Quarz'Up Chemin Delay, 7, 1214, Vernier, CH
  OK    25b rout

In [ ]:
# ─── Distance Summary: Original vs Optimised Route ───────────────────────────

def compute_route_distance(address_list, dist_matrix, index_map):
    """Sum consecutive leg distances for a given ordered list of addresses."""
    total = 0.0
    legs = []
    for i in range(len(address_list) - 1):
        a = index_map[address_list[i]]
        b = index_map[address_list[i + 1]]
        d = dist_matrix[a][b]
        legs.append((address_list[i], address_list[i + 1], d))
        total += d
    return total, legs

# Build a lookup from address string to matrix index
index_map = {addr: i for i, addr in enumerate(locations)}

# Original route = matrix order, returning to depot
original_order = locations + [locations[0]]

# Optimised route already ends at depot (appended by solver)
optimised_order = route

# Compute distances
original_total,   original_legs   = compute_route_distance(original_order,   distance_matrix, index_map)
optimised_total,  optimised_legs  = compute_route_distance(optimised_order,  distance_matrix, index_map)

# Saving
saving_km  = original_total  - optimised_total
saving_pct = (saving_km / original_total * 100) if original_total > 0 else 0

# ─── Print results ────────────────────────────────────────────────────────────
col = 60  # column width for address truncation

print("=" * 90)
print("ORIGINAL ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(original_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total original distance : {original_total:.3f} km")

print()
print("=" * 90)
print("OPTIMISED ROUTE")
print("=" * 90)
for i, (src, dst, d) in enumerate(optimised_legs):
    print(f"  {i+1:>2}. {src[:col]:<{col}}  ->  {dst[:col]:<{col}}  {d:>8.3f} km")
print(f"\n  Total optimised distance: {optimised_total:.3f} km")

print()
print("=" * 90)
print(f"  Saving : {saving_km:.3f} km  ({saving_pct:.1f}% reduction)")
print("=" * 90)

ORIGINAL ROUTE
   1. Rue Blavignac 17, 1227, Carouge, CH                           ->  Ch. de la Place-Verte 34, 1234, Vessy, CH                        3.064 km
   2. Ch. de la Place-Verte 34, 1234, Vessy, CH                     ->  Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH               5.490 km
   3. Chemin du Pré-Fleuri 15, 1228, Plan-les-Ouates, CH            ->  Chemin Charles-Borgeaud 27, 1213, Onex, CH                       4.579 km
   4. Chemin Charles-Borgeaud 27, 1213, Onex, CH                    ->  Rte de Chancy 85, 1213, Onex, CH                                 1.222 km
   5. Rte de Chancy 85, 1213, Onex, CH                              ->  Avenue des Morgines, 8, 1213, Genève, CH                         1.003 km
   6. Avenue des Morgines, 8, 1213, Genève, CH                      ->  Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH                  3.117 km
   7. Chemin du Chateau-Bloch 11, 1219, Le Lignon, CH               ->  Avenue de l’Etang 55, 1219, Châtelain

## Combining two routes to one for a VRP

## ORS model

In [ ]:
df = pd.read_excel("joint_distance_matrix.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 47 rows x 48 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Rue Blavignac, 10, 1227, Carouge, CH","Rte de Saint Julien 7, 1227, Carouge GE, CH","Rue Joseph Girard 18, 1227, Carouge GE, CH","Clos de la Fonderie 1, 1227, Carouge GE, CH","Avenue de la Praille 26, 1227, Genève, CH","Avenue de la Praille 50, 1227, Carouge GE, CH","Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH","Rte des Jeunes 9, 1227, Les Acacias, CH",...,"Chemin des mines 9, 1203, Genève, CH","Avenue de Sécheron 15, 1202, Geneva, CH","Avenue de Secheron 6, 1202, Genève, CH","Chemin Eugène-Rigot 2, 1202, Genève, CH","Rue de Vermont 62, 1202, Genève, CH","Avenue de la paix 19, 1202, Genève, CH","Avenue Louis-Casai 18, 1209, Genève, CH","Rue de Bourgogne 23, 1203, Genève, CH","Rue de Saint-Jean 30, 1203, Genève, CH","Place de Pont-Rouge, 1212 Lancy, Suisse"
0,"Rue Blavignac 17, 1227, ,",0.00,0.23,1.57,1.32,1.40,0.94,1.46,2.50,1.60,...,5.42,5.24,5.20,5.57,5.33,6.39,6.15,4.82,4.10,1.98
1,"Rue Blavignac, 10, 1227, Carouge, CH",0.19,0.00,1.58,1.33,1.40,0.94,1.46,2.51,1.60,...,5.43,5.24,5.20,5.57,5.33,6.40,6.16,4.82,4.10,1.98
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",1.21,1.03,0.00,0.69,1.71,1.42,1.94,1.69,2.07,...,5.76,5.57,5.53,5.91,5.66,6.73,6.49,5.15,4.43,2.46
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",1.16,0.97,1.12,0.00,1.05,1.29,1.81,2.05,1.95,...,5.45,5.27,5.23,5.60,5.36,6.43,6.36,5.02,4.30,2.33
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",1.68,1.50,2.25,1.60,0.00,0.87,1.38,3.21,1.52,...,4.53,4.34,4.30,4.67,4.43,5.50,5.45,4.11,3.39,1.90


In [ ]:
def load_distance_matrix(file_path):
    df = pd.read_excel(file_path, index_col=0)
    matrix = df.to_numpy().tolist()
    labels = df.index.astype(str).tolist()

    n = len(matrix)
    if any(len(row) != n for row in matrix):
        raise ValueError("Distance matrix must be square")

    return matrix, labels, df

In [ ]:
def solve_balanced_vrp(distance_matrix, depot=0, num_vehicles=2, scale=1000):

    n = len(distance_matrix)

    # Scale and round instead of truncating.
    # OR-Tools expects integer transit costs.
    scaled_matrix = [
        [int(round(d * scale)) for d in row]
        for row in distance_matrix
    ]

    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return scaled_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Minimize total traveled distance.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Add a distance dimension so we can balance route lengths.
    dimension_name = "Distance"
    max_possible_distance = sum(max(row) for row in scaled_matrix)

    routing.AddDimension(
        transit_callback_index,
        0,                    # no slack
        max_possible_distance,  # large enough upper bound
        True,                 # start cumul to zero
        dimension_name
    )

    distance_dimension = routing.GetDimensionOrDie(dimension_name)

    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 10

    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        return None, None

    routes = []
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_nodes = []
        route_distance_scaled = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            next_index = solution.Value(routing.NextVar(index))
            route_distance_scaled += routing.GetArcCostForVehicle(index, next_index, vehicle_id)
            index = next_index

        route_nodes.append(manager.IndexToNode(index))

        routes.append({
            "vehicle_id": vehicle_id,
            "route": route_nodes,
            "distance_scaled": route_distance_scaled,
            "distance_actual": route_distance_scaled / scale
        })

    return routes, solution.ObjectiveValue()

In [ ]:
def geocode_addresses(labels, country_hint=None):
    geolocator = Nominatim(user_agent="colab_vrp_mapper")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    records = []
    for label in labels:
        query = f"{label}, {country_hint}" if country_hint else label
        location = geocode(query)

        records.append({
            "address": label,
            "latitude": location.latitude if location else None,
            "longitude": location.longitude if location else None,
            "matched_address": location.address if location else None
        })

    return pd.DataFrame(records)

In [ ]:
def build_route_map_inline(routes, labels, geocoded_df):
    valid_points = geocoded_df.dropna(subset=["latitude", "longitude"])
    if valid_points.empty:
        raise ValueError("No valid geocoded addresses found.")

    center = [
        valid_points["latitude"].mean(),
        valid_points["longitude"].mean()
    ]

    m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")

    coords_lookup = geocoded_df.set_index("address")[["latitude", "longitude"]].to_dict("index")
    colors = ["blue", "red", "green", "purple", "orange"]
    all_bounds = []

    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        color = colors[vehicle_id % len(colors)]

        route_coords = []

        for stop_order, node_idx in enumerate(route_nodes):
            address = labels[node_idx]
            lat = coords_lookup[address]["latitude"]
            lon = coords_lookup[address]["longitude"]

            if pd.notna(lat) and pd.notna(lon):
                route_coords.append([lat, lon])
                all_bounds.append([lat, lon])

                folium.Marker(
                    location=[lat, lon],
                    tooltip=f"Vehicle {vehicle_id} - Stop {stop_order}",
                    popup=f"""
                    <b>Vehicle:</b> {vehicle_id}<br>
                    <b>Stop:</b> {stop_order}<br>
                    <b>Address:</b> {address}
                    """,
                    icon=folium.Icon(color=color)
                ).add_to(m)

        if len(route_coords) >= 2:
            folium.PolyLine(
                route_coords,
                color=color,
                weight=5,
                opacity=0.85,
                tooltip=f"Route for vehicle {vehicle_id}"
            ).add_to(m)

    if all_bounds:
        m.fit_bounds(all_bounds)

    return m

In [ ]:
def prepare_route_table(routes, labels):
    route_summary = []
    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        total_distance = route_info["distance_actual"]

        route_summary.append({
            "vehicle_id": vehicle_id,
            "route": " -> ".join([labels[n] for n in route_nodes]),
            "num_stops": len(route_nodes),
            "total_distance": total_distance
        })
    return pd.DataFrame(route_summary)

In [ ]:
# ==== RUN EVERYTHING ====
file_path = "joint_distance_matrix.xlsx"
depot = 0
country_hint = "Switzerland"

distance_matrix, labels, raw_df = load_distance_matrix(file_path)
import numpy as np

result = solve_balanced_vrp(distance_matrix, depot=depot, num_vehicles=2)

if result is None:
    print("No solution found.")
else:
    routes, routes_df = result

    geocoded_df = geocode_addresses(labels, country_hint=country_hint)
    route_table = prepare_route_table(routes, labels)

    display(route_table)
    display(geocoded_df)


,vehicle_id,route,num_stops,total_distance
0,0,"Rue Blavignac 17, 1227, , -> Rue François-Duss...",26,21.45
1,1,"Rue Blavignac 17, 1227, , -> Avenue de la prai...",24,21.47


,address,latitude,longitude,matched_address
0,"Rue Blavignac 17, 1227, ,",46.182138,6.134618,"Rue Blavignac, Pinchat, La Praille, Carouge, G..."
1,"Rue Blavignac, 10, 1227, Carouge, CH",46.182009,6.133665,"10, Rue Blavignac, Pinchat, La Praille, Caroug..."
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",46.179767,6.138165,"Route de Saint-Julien, Pinchat, La Praille, Ca..."
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",46.180695,6.142035,"18, Rue Joseph-Girard, Pinchat, La Praille, Ca..."
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",46.186363,6.143332,"1a, Clos-de-la-Fonderie, La Praille, Carouge, ..."
5,"Avenue de la Praille 26, 1227, Genève, CH",46.187344,6.135355,"26, Avenue de la Praille, La Praille, Carouge,..."
6,"Avenue de la praille 50, 1227, Carouge GE, CH",46.187029,6.128741,"50, Avenue de la Praille, La Praille, Carouge,..."
7,"Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH",46.186735,6.125885,"Esplanade de Pont-Rouge, Grand-Lancy, Lancy, G..."
8,"Rte des Jeunes 9, 1227, Les Acacias, CH",46.187960,6.128525,"McDonald's, 9, Route des Jeunes, Les Acacias, ..."
9,"Passage de la Radio, 1211, Genève 1, CH",NaN,NaN,None


In [ ]:
route_map = build_route_map_inline(routes, labels, geocoded_df)
display(route_map)

In [ ]:
def solve_shortest_total_distance_vrp(distance_matrix, depot=0, num_vehicles=2, scale=1000):

    n = len(distance_matrix)

    # Keep the actual distances by rounding after scaling.
    scaled_matrix = [
        [int(round(d * scale)) for d in row]
        for row in distance_matrix
    ]

    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return scaled_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Objective: minimize total arc cost across all vehicles.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Optional distance dimension, useful for reporting route lengths.
    routing.AddDimension(
        transit_callback_index,
        0,  # slack
        sum(sum(row) for row in scaled_matrix),  # large enough upper bound
        True,  # start cumul at zero
        "Distance"
    )

    distance_dimension = routing.GetDimensionOrDie("Distance")

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 10

    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        return None, None

    routes = []
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route = []
        route_distance_scaled = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
            next_index = solution.Value(routing.NextVar(index))
            route_distance_scaled += routing.GetArcCostForVehicle(index, next_index, vehicle_id)
            index = next_index

        route.append(manager.IndexToNode(index))

        routes.append({
            "vehicle_id": vehicle_id,
            "route": route,
            "distance_scaled": route_distance_scaled,
            "distance_actual": route_distance_scaled / scale
        })

    return routes, solution.ObjectiveValue() / scale


In [ ]:
def geocode_addresses(labels, country_hint=None):
    geolocator = Nominatim(user_agent="colab_vrp_mapper")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    records = []
    for label in labels:
        query = f"{label}, {country_hint}" if country_hint else label
        location = geocode(query)

        records.append({
            "address": label,
            "latitude": location.latitude if location else None,
            "longitude": location.longitude if location else None,
            "matched_address": location.address if location else None
        })

    return pd.DataFrame(records)

In [ ]:
def build_route_map_inline(routes, labels, geocoded_df):
    valid_points = geocoded_df.dropna(subset=["latitude", "longitude"])
    if valid_points.empty:
        raise ValueError("No valid geocoded addresses found.")

    center = [
        valid_points["latitude"].mean(),
        valid_points["longitude"].mean()
    ]

    m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")

    coords_lookup = geocoded_df.set_index("address")[["latitude", "longitude"]].to_dict("index")
    colors = ["blue", "red", "green", "purple", "orange"]
    all_bounds = []

    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        color = colors[vehicle_id % len(colors)]

        route_coords = []

        for stop_order, node_idx in enumerate(route_nodes):
            address = labels[node_idx]
            lat = coords_lookup[address]["latitude"]
            lon = coords_lookup[address]["longitude"]

            if pd.notna(lat) and pd.notna(lon):
                route_coords.append([lat, lon])
                all_bounds.append([lat, lon])

                folium.Marker(
                    location=[lat, lon],
                    tooltip=f"Vehicle {vehicle_id} - Stop {stop_order}",
                    popup=f"""
                    <b>Vehicle:</b> {vehicle_id}<br>
                    <b>Stop:</b> {stop_order}<br>
                    <b>Address:</b> {address}
                    """,
                    icon=folium.Icon(color=color)
                ).add_to(m)

        if len(route_coords) >= 2:
            folium.PolyLine(
                route_coords,
                color=color,
                weight=5,
                opacity=0.85,
                tooltip=f"Route for vehicle {vehicle_id}"
            ).add_to(m)

    if all_bounds:
        m.fit_bounds(all_bounds)

    return m

In [ ]:
def prepare_route_table(routes, labels):
    route_summary = []
    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        total_distance = route_info["distance_actual"]

        route_summary.append({
            "vehicle_id": vehicle_id,
            "route": " -> ".join([labels[n] for n in route_nodes]),
            "num_stops": len(route_nodes),
            "total_distance": total_distance
        })
    return pd.DataFrame(route_summary)

In [ ]:
file_path = "joint_distance_matrix.xlsx"
depot = 0
country_hint = "Switzerland"

distance_matrix, labels, raw_df = load_distance_matrix(file_path)
import numpy as np

result = solve_shortest_total_distance_vrp(distance_matrix, depot=depot, num_vehicles=2, scale=1000)

if result is None:
    print("No solution found.")
else:
    routes, routes_df = result

    geocoded_df = geocode_addresses(labels, country_hint=country_hint)
    route_table = prepare_route_table(routes, labels)

    display(route_table)
    display(geocoded_df)

,vehicle_id,route,num_stops,total_distance
0,0,"Rue Blavignac 17, 1227, , -> Rue Blavignac 17,...",2,0.0
1,1,"Rue Blavignac 17, 1227, , -> Route des Acacias...",48,36.6


,address,latitude,longitude,matched_address
0,"Rue Blavignac 17, 1227, ,",46.182138,6.134618,"Rue Blavignac, Pinchat, La Praille, Carouge, G..."
1,"Rue Blavignac, 10, 1227, Carouge, CH",46.182009,6.133665,"10, Rue Blavignac, Pinchat, La Praille, Caroug..."
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",46.179575,6.138148,"Route de Saint-Julien, Pinchat, La Praille, Ca..."
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",46.180695,6.142035,"18, Rue Joseph-Girard, Pinchat, La Praille, Ca..."
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",46.186363,6.143332,"1a, Clos-de-la-Fonderie, La Praille, Carouge, ..."
5,"Avenue de la Praille 26, 1227, Genève, CH",46.187344,6.135355,"26, Avenue de la Praille, La Praille, Carouge,..."
6,"Avenue de la praille 50, 1227, Carouge GE, CH",46.187376,6.128943,"Clim Diffusion SA, 50, Avenue de la Praille, L..."
7,"Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH",46.186735,6.125885,"Esplanade de Pont-Rouge, Grand-Lancy, Lancy, G..."
8,"Rte des Jeunes 9, 1227, Les Acacias, CH",46.187960,6.128525,"McDonald's, 9, Route des Jeunes, Les Acacias, ..."
9,"Passage de la Radio, 1211, Genève 1, CH",NaN,NaN,None


In [ ]:
route_map = build_route_map_inline(routes, labels, geocoded_df)
display(route_map)

## HERE model

In [39]:
df = pd.read_excel("joint_distance_matrix_2.xlsx")
print(f"Dataset loaded -- {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Dataset loaded -- 47 rows x 48 columns


,DISTANCE MATRIX,"Rue Blavignac 17, 1227, ,","Rue Blavignac, 10, 1227, Carouge, CH","Rte de Saint Julien 7, 1227, Carouge GE, CH","Rue Joseph Girard 18, 1227, Carouge GE, CH","Clos de la Fonderie 1, 1227, Carouge GE, CH","Avenue de la Praille 26, 1227, Genève, CH","Avenue de la Praille 50, 1227, Carouge GE, CH","Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH","Rte des Jeunes 9, 1227, Les Acacias, CH",...,"Chemin des mines 9, 1203, Genève, CH","Avenue de Sécheron 15, 1202, Geneva, CH","Avenue de Secheron 6, 1202, Genève, CH","Chemin Eugène-Rigot 2, 1202, Genève, CH","Rue de Vermont 62, 1202, Genève, CH","Avenue de la paix 19, 1202, Genève, CH","Avenue Louis-Casai 18, 1209, Genève, CH","Rue de Bourgogne 23, 1203, Genève, CH","Rue de Saint-Jean 30, 1203, Genève, CH","Place de Pont-Rouge, 1212 Lancy, CH"
0,"Rue Blavignac 17, 1227, ,",0.000,0.213,1.130,1.414,1.364,1.322,1.274,1.545,1.440,...,5.333,5.170,5.223,5.334,5.255,6.235,6.020,4.961,3.595,2.161
1,"Rue Blavignac, 10, 1227, Carouge, CH",0.295,0.000,1.045,1.249,1.199,1.102,1.065,1.336,1.231,...,5.168,5.005,5.058,5.169,5.090,6.070,5.855,4.577,3.386,1.952
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",1.158,1.136,0.000,0.662,1.436,1.802,1.920,2.191,2.086,...,5.584,5.421,5.474,5.585,5.506,6.486,6.271,5.212,3.868,2.807
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",1.085,1.063,1.123,0.000,1.028,1.561,1.671,1.942,1.837,...,5.188,5.025,5.078,5.189,5.110,6.090,5.875,4.816,4.149,2.558
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",1.541,1.309,2.162,0.990,0.000,1.187,1.365,1.636,1.531,...,4.495,4.332,4.385,4.496,4.417,5.397,5.182,4.123,2.851,2.346


In [40]:
def load_distance_matrix(file_path):
    df = pd.read_excel(file_path, index_col=0)
    matrix = df.to_numpy().tolist()
    labels = df.index.astype(str).tolist()

    n = len(matrix)
    if any(len(row) != n for row in matrix):
        raise ValueError("Distance matrix must be square")

    return matrix, labels, df

In [41]:
def solve_vrp(distance_matrix, depot=0, num_vehicles=2, scale=1000):

    n = len(distance_matrix)

    # Scale and round instead of truncating.
    # OR-Tools expects integer transit costs.
    scaled_matrix = [
        [int(round(d * scale)) for d in row]
        for row in distance_matrix
    ]

    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return scaled_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    dimension_name = "Distance"
    max_possible_distance = sum(max(row) for row in scaled_matrix)

    routing.AddDimension(
        transit_callback_index,
        0,                    #
        max_possible_distance,
        True,
        dimension_name
    )

    distance_dimension = routing.GetDimensionOrDie(dimension_name)

    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 10

    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        return None, None

    routes = []
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_nodes = []
        route_distance_scaled = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route_nodes.append(node)
            next_index = solution.Value(routing.NextVar(index))
            route_distance_scaled += routing.GetArcCostForVehicle(index, next_index, vehicle_id)
            index = next_index

        route_nodes.append(manager.IndexToNode(index))

        routes.append({
            "vehicle_id": vehicle_id,
            "route": route_nodes,
            "distance_scaled": route_distance_scaled,
            "distance_actual": route_distance_scaled / scale
        })

    return routes, solution.ObjectiveValue()

In [42]:
def geocode_addresses(labels, country_hint=None):
    geolocator = Nominatim(user_agent="colab_vrp_mapper")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    records = []
    for label in labels:
        query = f"{label}, {country_hint}" if country_hint else label
        location = geocode(query)

        records.append({
            "address": label,
            "latitude": location.latitude if location else None,
            "longitude": location.longitude if location else None,
            "matched_address": location.address if location else None
        })

    return pd.DataFrame(records)

In [43]:
def build_route_map_inline(routes, labels, geocoded_df):
    valid_points = geocoded_df.dropna(subset=["latitude", "longitude"])
    if valid_points.empty:
        raise ValueError("No valid geocoded addresses found.")

    center = [
        valid_points["latitude"].mean(),
        valid_points["longitude"].mean()
    ]

    m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")

    coords_lookup = geocoded_df.set_index("address")[["latitude", "longitude"]].to_dict("index")
    colors = ["blue", "red", "green", "purple", "orange"]
    all_bounds = []

    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        color = colors[vehicle_id % len(colors)]

        route_coords = []

        for stop_order, node_idx in enumerate(route_nodes):
            address = labels[node_idx]
            lat = coords_lookup[address]["latitude"]
            lon = coords_lookup[address]["longitude"]

            if pd.notna(lat) and pd.notna(lon):
                route_coords.append([lat, lon])
                all_bounds.append([lat, lon])

                folium.Marker(
                    location=[lat, lon],
                    tooltip=f"Vehicle {vehicle_id} - Stop {stop_order}",
                    popup=f"""
                    <b>Vehicle:</b> {vehicle_id}<br>
                    <b>Stop:</b> {stop_order}<br>
                    <b>Address:</b> {address}
                    """,
                    icon=folium.Icon(color=color)
                ).add_to(m)

        if len(route_coords) >= 2:
            folium.PolyLine(
                route_coords,
                color=color,
                weight=5,
                opacity=0.85,
                tooltip=f"Route for vehicle {vehicle_id}"
            ).add_to(m)

    if all_bounds:
        m.fit_bounds(all_bounds)

    return m

In [44]:
def prepare_route_table(routes, labels):
    route_summary = []
    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        total_distance = route_info["distance_actual"]

        route_summary.append({
            "vehicle_id": vehicle_id,
            "route": " -> ".join([labels[n] for n in route_nodes]),
            "num_stops": len(route_nodes),
            "total_distance": total_distance
        })
    return pd.DataFrame(route_summary)

In [45]:
# ==== RUN EVERYTHING ====
file_path = "joint_distance_matrix_2.xlsx"
depot = 0
country_hint = "Switzerland"

distance_matrix, labels, raw_df = load_distance_matrix(file_path)
import numpy as np

result = solve_vrp(distance_matrix, depot=depot, num_vehicles=2)

if result is None:
    print("No solution found.")
else:
    routes, routes_df = result

    geocoded_df = geocode_addresses(labels, country_hint=country_hint)
    route_table = prepare_route_table(routes, labels)

    display(route_table)
    display(geocoded_df)

,vehicle_id,route,num_stops,total_distance
0,0,"Rue Blavignac 17, 1227, , -> Rue Blavignac, 10...",30,19.460
1,1,"Rue Blavignac 17, 1227, , -> Avenue de la prai...",20,20.305


,address,latitude,longitude,matched_address
0,"Rue Blavignac 17, 1227, ,",46.182138,6.134618,"Rue Blavignac, Pinchat, La Praille, Carouge, G..."
1,"Rue Blavignac, 10, 1227, Carouge, CH",46.182009,6.133665,"10, Rue Blavignac, Pinchat, La Praille, Caroug..."
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",46.179767,6.138165,"Route de Saint-Julien, Pinchat, La Praille, Ca..."
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",46.180695,6.142035,"18, Rue Joseph-Girard, Pinchat, La Praille, Ca..."
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",46.186363,6.143332,"1a, Clos-de-la-Fonderie, La Praille, Carouge, ..."
5,"Avenue de la Praille 26, 1227, Genève, CH",46.187344,6.135355,"26, Avenue de la Praille, La Praille, Carouge,..."
6,"Avenue de la praille 50, 1227, Carouge GE, CH",46.187029,6.128741,"50, Avenue de la Praille, La Praille, Carouge,..."
7,"Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH",46.186735,6.125885,"Esplanade de Pont-Rouge, Grand-Lancy, Lancy, G..."
8,"Rte des Jeunes 9, 1227, Les Acacias, CH",46.187960,6.128525,"McDonald's, 9, Route des Jeunes, Les Acacias, ..."
9,"Passage de la Radio, 1211, Genève 1, CH",NaN,NaN,None


In [46]:
route_map = build_route_map_inline(routes, labels, geocoded_df)
display(route_map)

In [47]:
def solve_shortest_total_distance_vrp(distance_matrix, depot=0, num_vehicles=2, scale=1000):

    n = len(distance_matrix)

    # Keep the actual distances by rounding after scaling.
    scaled_matrix = [
        [int(round(d * scale)) for d in row]
        for row in distance_matrix
    ]

    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return scaled_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)

    # Objective: minimize total arc cost across all vehicles.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Optional distance dimension, useful for reporting route lengths.
    routing.AddDimension(
        transit_callback_index,
        0,  # slack
        sum(sum(row) for row in scaled_matrix),  # large enough upper bound
        True,  # start cumul at zero
        "Distance"
    )

    distance_dimension = routing.GetDimensionOrDie("Distance")

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.seconds = 30

    solution = routing.SolveWithParameters(search_parameters)

    if not solution:
        return None, None

    routes = []
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route = []
        route_distance_scaled = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(node)
            next_index = solution.Value(routing.NextVar(index))
            route_distance_scaled += routing.GetArcCostForVehicle(index, next_index, vehicle_id)
            index = next_index

        route.append(manager.IndexToNode(index))

        routes.append({
            "vehicle_id": vehicle_id,
            "route": route,
            "distance_scaled": route_distance_scaled,
            "distance_actual": route_distance_scaled / scale
        })

    return routes, solution.ObjectiveValue() / scale

In [48]:
def geocode_addresses(labels, country_hint=None):
    geolocator = Nominatim(user_agent="colab_vrp_mapper")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

    records = []
    for label in labels:
        query = f"{label}, {country_hint}" if country_hint else label
        location = geocode(query)

        records.append({
            "address": label,
            "latitude": location.latitude if location else None,
            "longitude": location.longitude if location else None,
            "matched_address": location.address if location else None
        })

    return pd.DataFrame(records)

In [ ]:
def build_route_map_inline(routes, labels, geocoded_df):
    valid_points = geocoded_df.dropna(subset=["latitude", "longitude"])
    if valid_points.empty:
        raise ValueError("No valid geocoded addresses found.")

    center = [
        valid_points["latitude"].mean(),
        valid_points["longitude"].mean()
    ]

    m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")

    coords_lookup = geocoded_df.set_index("address")[["latitude", "longitude"]].to_dict("index")
    colors = ["blue", "red", "green", "purple", "orange"]
    all_bounds = []

    for route_info in routes:
        vehicle_id = route_info["vehicle_id"]
        route_nodes = route_info["route"]
        color = colors[vehicle_id % len(colors)]

        route_coords = []

        for stop_order, node_idx in enumerate(route_nodes):
            address = labels[node_idx]
            lat = coords_lookup[address]["latitude"]
            lon = coords_lookup[address]["longitude"]

            if pd.notna(lat) and pd.notna(lon):
                route_coords.append([lat, lon])
                all_bounds.append([lat, lon])

                folium.Marker(
                    location=[lat, lon],
                    tooltip=f"Vehicle {vehicle_id} - Stop {stop_order}",
                    popup=f"""
                    <b>Vehicle:</b> {vehicle_id}<br>
                    <b>Stop:</b> {stop_order}<br>
                    <b>Address:</b> {address}
                    """,
                    icon=folium.Icon(color=color)
                ).add_to(m)

        if len(route_coords) >= 2:
            folium.PolyLine(
                route_coords,
                color=color,
                weight=5,
                opacity=0.85,
                tooltip=f"Route for vehicle {vehicle_id}"
            ).add_to(m)

    if all_bounds:
        m.fit_bounds(all_bounds)

    return m

In [ ]:
file_path = "joint_distance_matrix_2.xlsx"
depot = 0
country_hint = "Switzerland"

distance_matrix, labels, raw_df = load_distance_matrix(file_path)
import numpy as np

result = solve_shortest_total_distance_vrp(distance_matrix, depot=depot, num_vehicles=2, scale=1000)

if result is None:
    print("No solution found.")
else:
    routes, routes_df = result

    geocoded_df = geocode_addresses(labels, country_hint=country_hint)
    route_table = prepare_route_table(routes, labels)

    display(route_table)
    display(geocoded_df)

,vehicle_id,route,num_stops,total_distance
0,0,"Rue Blavignac 17, 1227, , -> Rue Blavignac 17,...",2,0.000
1,1,"Rue Blavignac 17, 1227, , -> Rte de Saint Juli...",48,33.382


,address,latitude,longitude,matched_address
0,"Rue Blavignac 17, 1227, ,",46.182138,6.134618,"Rue Blavignac, Pinchat, La Praille, Carouge, G..."
1,"Rue Blavignac, 10, 1227, Carouge, CH",46.182009,6.133665,"10, Rue Blavignac, Pinchat, La Praille, Caroug..."
2,"Rte de Saint Julien 7, 1227, Carouge GE, CH",46.179575,6.138148,"Route de Saint-Julien, Pinchat, La Praille, Ca..."
3,"Rue Joseph Girard 18, 1227, Carouge GE, CH",46.180695,6.142035,"18, Rue Joseph-Girard, Pinchat, La Praille, Ca..."
4,"Clos de la Fonderie 1, 1227, Carouge GE, CH",46.186363,6.143332,"1a, Clos-de-la-Fonderie, La Praille, Carouge, ..."
5,"Avenue de la Praille 26, 1227, Genève, CH",46.187344,6.135355,"26, Avenue de la Praille, La Praille, Carouge,..."
6,"Avenue de la praille 50, 1227, Carouge GE, CH",46.187376,6.128943,"Clim Diffusion SA, 50, Avenue de la Praille, L..."
7,"Esplanade de Pont Rouge 9c, 1212, Grand Lancy, CH",46.186735,6.125885,"Esplanade de Pont-Rouge, Grand-Lancy, Lancy, G..."
8,"Rte des Jeunes 9, 1227, Les Acacias, CH",46.187960,6.128525,"McDonald's, 9, Route des Jeunes, Les Acacias, ..."
9,"Passage de la Radio, 1211, Genève 1, CH",NaN,NaN,None


In [ ]:
route_map = build_route_map_inline(routes, labels, geocoded_df)
display(route_map)